# 04 — NLP: Gemini-Assisted Review Labeling and Demand Signals

Weak supervision using Gemini Flash labels, aggregated into zone-level demand features — works for any cuisine concept.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
%matplotlib inline

## Sample Reviews — Any Cuisine


In [ ]:
# Reviews span many cuisine types to demonstrate generality
sample_reviews = [
    "Amazing healthy Indian lunch bowl — fresh and light, will return.",
    "Great tonkotsu ramen, rich broth, came back twice this week.",
    "Authentic Korean BBQ with top-quality bulgogi and kimchi.",
    "Best tacos al pastor in the neighborhood, super fresh.",
    "Mediterranean bowls are filling but could use more variety.",
    "Average burger, nothing special, would not recommend.",
    "Incredible Ethiopian injera platter — generous portions.",
    "Solid vegan wrap but the salad bar is a bit sparse.",
    "Dim sum was mediocre, har gow skin too thick.",
    "Fresh smoothie bowls, acai is amazing, staff friendly.",
    "Overpriced Italian pasta, nothing you cannot get elsewhere.",
    "Healthy Indian dosa was excellent — gluten-free and filling.",
]
print(f"{len(sample_reviews)} sample reviews")

## Gemini Label Generation (offline fallback when no API key)


In [ ]:
from src.nlp.gemini_labels import label_reviews_with_gemini, build_label_prompt
from src.utils.taxonomy import all_known_subtypes

subtypes = all_known_subtypes()
print(f"Total supported subtypes: {len(subtypes)}")
print(subtypes)

# Will use synthetic labels unless GEMINI_API_KEY is set
labels = label_reviews_with_gemini(sample_reviews, subtypes, api_key=None)
label_df = pd.DataFrame([vars(label) for label in labels])
label_df["review_text"] = sample_reviews
display(
    label_df[["review_id", "sentiment", "concept_subtype", "confidence", "review_text"]]
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3))

label_df["sentiment"].value_counts().plot.bar(
    ax=axes[0], color="#4C72B0", edgecolor="white"
)
axes[0].set_title("Sentiment Distribution")
axes[0].set_xlabel("Sentiment")

label_df["concept_subtype"].value_counts().head(10).plot.barh(
    ax=axes[1], color="#55A868", edgecolor="white"
)
axes[1].set_title("Concept Subtype Distribution (top-10)")
axes[1].set_xlabel("Count")

plt.tight_layout()
plt.show()

## Subtype Classification (keyword-based)


In [ ]:
from src.nlp.subtype_classifier import batch_classify

classified = batch_classify(sample_reviews)
comp_df = pd.DataFrame(
    {
        "review": sample_reviews,
        "gemini_label": label_df["concept_subtype"].values,
        "keyword_label": classified,
    }
)
comp_df["match"] = comp_df["gemini_label"] == comp_df["keyword_label"]
display(comp_df)
print(f"Agreement rate: {comp_df['match'].mean():.0%}")

## Review Aggregation into Zone Features


In [ ]:
from src.nlp.review_aggregates import aggregate_review_labels

rng = np.random.default_rng(42)
zones = ["bk-tandon", "mn-columbia", "qn-astoria", "bx-fordham", "si-st-george"]
label_df["zone_id"] = rng.choice(zones, size=len(label_df))
label_df["time_key"] = rng.choice([2022, 2023, 2024], size=len(label_df))

agg = aggregate_review_labels(label_df)
print("Aggregated zone-time features:")
display(agg)

## White-Space Gap per Subtype


In [ ]:
from src.nlp.white_space import compute_subtype_gap

# Simulate demand vs supply per subtype across a sample zone
rng2 = np.random.default_rng(7)
all_subtypes = list(all_known_subtypes())
demand = rng2.uniform(0.3, 0.9, size=len(all_subtypes))
supply = rng2.uniform(0.1, 0.7, size=len(all_subtypes))
gaps = [compute_subtype_gap(d, s) for d, s in zip(demand, supply)]

gap_df = pd.DataFrame(
    {"subtype": all_subtypes, "demand": demand, "supply": supply, "gap": gaps}
)
gap_df = gap_df.sort_values("gap", ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(gap_df["subtype"], gap_df["gap"], color="#C44E52", edgecolor="white")
ax.set_xticklabels(gap_df["subtype"], rotation=45, ha="right", fontsize=9)
ax.set_ylabel("White-space gap score")
ax.set_title("Cuisine Gap Scores — Sample NYC Zone (any concept type)")
plt.tight_layout()
plt.show()

## Prompt Example


In [ ]:
prompt = build_label_prompt(
    sample_reviews[0], ("healthy_indian", "ramen", "korean", "mexican")
)
print(prompt)

## Summary

- The NLP pipeline supports any cuisine concept, not just healthy food.
- Without an API key, synthetic labels are produced for offline development.
- With `GEMINI_API_KEY` set, real Gemini 2.5 Flash Lite labels are generated.
- Zone-level `healthy_review_share` and `subtype_gap` feed directly into the CMF scoring layer.
- Gap scores vary dramatically by cuisine — some subtypes are heavily under-supplied.
